# GardnAIze — 04 Explore and validate garden images

Objective: build an auditable labeled dataset for **garden**, **not_garden**, **unusable**. This notebook does not download or redistribute images. Confirm each image's license and provenance before use.

Expected layout: `datasets/images/garden_validation/{garden,not_garden,unusable}/<images>`; manually label images; do not infer soil type from appearance.

In [ ]:
import pandas as pd
from PIL import Image, UnidentifiedImageError
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "datasets" / "images" / "garden_validation"
CLASSES = ["garden", "not_garden", "unusable"]
print("Dataset:", DATA)

records = []
for label in CLASSES:
    for path in sorted((DATA / label).glob("*")):
        if not path.is_file():
            continue
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                width, height = im.size
            records.append({"path":str(path),"label":label,"width":width,"height":height,"valid":True})
        except (UnidentifiedImageError, OSError, ValueError):
            records.append({"path":str(path),"label":label,"width":None,"height":None,"valid":False})
df = pd.DataFrame(records,columns=["path","label","width","height","valid"])
display(df.groupby(["label","valid"]).size().rename("images").reset_index()) if not df.empty else print("No labeled images found. Add licensed images first.")

## Provenance and leakage controls

Maintain a separate local CSV with source URL, author, license, retrieval date, image hash, labeler and group identifier (same garden / same photographer). Split by group to prevent near-duplicate leakage. Do not use plant species API data as garden-scene image labels.

In [ ]:
import hashlib
if not df.empty:
    df["sha256"] = df["path"].map(lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest())
    print("Duplicate image hashes:", int(df["sha256"].duplicated().sum()))
    display(df.head())

Next: review class balance, label ambiguity and image licenses before training. `unusable` needs examples of blur, extreme darkness and low resolution, not simply non-garden photos.